In [5]:
import sys
sys.path.append('..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [6]:
def transform_graduate_state_data(file_path):
    df = pd.read_csv(file_path)
    
    # reshape years into rows
    df_long = df.melt(
        id_vars=["state", "statistics"],
        var_name="year",
        value_name="value"
    )
    
    # pivot statistics into columns
    df_pivot = df_long.pivot_table(
        index=["state", "year"],
        columns="statistics",
        values="value",
        aggfunc="first"
    ).reset_index()
    
    # flatten column names
    df_pivot.columns.name = None
    
    # convert numeric columns
    for col in df_pivot.columns:
        if col not in ["state", "year"] and "unemp_rate" not in col:
            df_pivot[col] = (
                df_pivot[col]
                .astype(str)
                .str.replace(",", "")
                .astype(float) * 1000
            )
        elif "unemp_rate" in col:
            df_pivot[col] = df_pivot[col].astype(float)
    
    return df_pivot

In [7]:
df_johor = transform_graduate_state_data("../../data/Johor_graduates.csv")
df_johor.head(10)

,state,year,deg_emp_graduate,deg_graduate,deg_outside_labour,deg_unemp_graduate,deg_unemp_rate,dip_emp_graduate,dip_graduate,dip_outside_labour,dip_unemp_graduate,dip_unemp_rate,total_emp_graduate,total_graduate,total_outside_labour,total_unemp_graduate,total_unemp_rate
0,Johor,2020,183500.0,207400.0,13600.0,10400.0,5.4,202500.0,243100.0,33000.0,7600.0,3.6,386000.0,450600.0,46600.0,18000.0,4.5
1,Johor,2021,196100.0,220700.0,13800.0,10800.0,5.2,213600.0,252700.0,32400.0,6600.0,3.0,409800.0,473400.0,46200.0,17400.0,4.1
2,Johor,2022,208300.0,235200.0,17000.0,10000.0,4.6,221700.0,263400.0,34300.0,7400.0,3.2,430000.0,498600.0,51300.0,17400.0,3.9
3,Johor,2023,224400.0,250400.0,16900.0,9100.0,3.9,230100.0,271300.0,35200.0,6100.0,2.6,454400.0,521700.0,52100.0,15200.0,3.2
4,Johor,2024,241900.0,272100.0,19800.0,10400.0,4.1,238800.0,282700.0,38100.0,5900.0,2.4,480700.0,554900.0,57900.0,16200.0,3.3
